INTRO PARAGRAPH GOES HERE

Plan:
* What is Spark?
  * Spark has many different feautres, but RDDs are fundamental
  * RDDs versus standard processing
  * Different APIs
* Dataframes
  * Way to structure data similar to a spreadsheet (columns, rows)
  * Introduced in API version ...
  * RDDs under the hood (Spark manages complexity)
* Using Spark Dataframes
  * Basic example

## Spark

Spark is an open-source framework used for processing large amounts of data quickly and reliably. 

The Spark engine is written in Scala. To interface with the Spark engine there are several supported APIs in different languages, including Java, Scale, Python and R. For this tutorial we'll be using the Python Spark API, known as PySpark.



## Processing Data

Before we go into the details of Spark, let's take a look at some of the issues that it solves.

When processing data a number of transformations are often applied in order to get it into a form that is useful. This can include transformations like filtering, grouping and mapping.

For example take a dashboard that displays sales information for the past week. Each sale is stored as a single record in a database table. It will include the date, cost, shipping and customer for each record. By transforming the data, records can be grouped by date, with the sum of sales calculated for each date. 

In a system where only one computer is used processing this data might look like this:

![Data Flow](images/spark-intro/etl_1.svg)


Data is fed in at one end (in this case sales data), a bunch of transformations are applied to the data in the middle, and then the data is output at the other end (in our example it is sent to a reporting dashboard). This system is often referred to as Extract, Transform, Load (ETL).

There are a few limitations to this approach of using a single machine to process large datasets:
1. Data is often processed synchronously. For large sets of data this can be slow.
1. When data is processed in parallel or asynchronously, this can lead to complex code or unforseen bugs like race conditions.
1. The system is not very fault tolerant. If the single computer crashes, the progress made transforming the data is lost. 

To overcome these issues, Spark uses a techique called a Resilient Distributed Dataset (RDD).

### Resilient Distributed Datasets

As with a standard ETL system, Spark takes a data input, transforms the data, and then loads it to another destination. The most significant difference is that Spark does transformation across a cluster of computers, instead of just one.

To process the data across a cluster of computers, Spark breaks the datasets into chunks. This allows the data to be processed in parallel on several machines, after which it is recombined into a single dataset. This technique is called a Resilient Distributed Dataset (RDD) and is fundamental to how Spark works.

Here is a modified version of the ETL diagram that shows the data being split into chunks and distributed:

![Data Flow](images/spark-intro/rdd_1.svg)

By chunking and distributing data in this way, Spark is able to process large datasets very quickly. It also means that it can scale up as the data gets larger by adding more computers to the cluster.

In addition to chunking and distributing data, Spark also does one other thing to the data in an RDD: it duplicates it. Before distributing the data across the cluster, Spark duplicates each chunk. The reason it does this is to protect against computers in the cluster failing. If a computer crashes while it is processing a chunk of data, it doesn't matter, the same chunk of data can be sent to be processed by another healthy computer in the cluster. 

By duplicating data with RDDs, the system is much more resilient to failures.


### Dataframes

Now that we know the concepts behind processing data with Spark's RDDs, lets look at how to use them.

Spark uses Dataframes to represent data. Dataframes are structured like a spreadsheet: they have a rows for data items and columns for each attribute of the data. Transformations can be applied to data directly from the dataframe using a verity of methods including filtering, grouping, agrregation and mapping.

Dataframes use RDD features under the hood. This means that the transformations applied to dataframes can be done across a cluster of computers, taking advantage of the speed and reslience of RDDs. Spark manages all of this complexity, allowing the programmer to define how they want the data to be processed, without worrying about distributing the data across a cluster of computers.

As well as running Spark on a cluster, we're able to run Spark on a single computer without any differences to the code. This means that you can develop your Spark code locally before you run it on a cluster in production.

Let's take a look using Dataframes in PySpark.

## PySpark

For this example I'll be using some fake sales data stored in a csv. I've made [the data available to download](/static/sales_data.csv) if you want to follow along with the example.


To get started with Spark's Python API you need to install PySpark:

```
pip install pyspark
```

Once `pyspark` is successfully installed we can start using Spark.

First we need to import pyspark and set up a spark session:


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('sales').getOrCreate()

This will start a new Spark app, or get an existing one if it already exists. I've named my Spark app `sales` in this case.

Now that I've started the session I can create a dataframe with my sales data:

In [2]:
sales_data = spark.read.csv('static/sales_data.csv', header=True,
                            inferSchema=True)

Spark comes with several methods to read common file types. Here I've used the `csv()` read method. There are also methods for json and text files, among many others.

The `inferSchema=True` argument means that Spark will automatically infer the data types for each column when the data is loaded. The `header=True` argument tells spark that the first row of the data contains the headers for each column.

Now that I've loaded the data into a dataframe, lets see what it looks like using the `show()` method:

In [3]:
sales_data.show()

+-------------------+----------+-----+-------------+
|               date|total_cost|items|     location|
+-------------------+----------+-----+-------------+
|2018-01-01 00:00:00|     15.71|    2|United States|
|2018-01-06 00:00:00|    148.13|    5|United States|
|2018-01-06 00:00:00|     54.28|    5|        China|
|2018-01-03 00:00:00|     92.39|    2|United States|
|2018-01-01 00:00:00|     16.73|    6|        China|
|2018-01-02 00:00:00|    172.91|    1|        China|
|2018-01-05 00:00:00|     69.54|    3|        China|
|2018-01-02 00:00:00|      21.7|    6|        China|
|2018-01-05 00:00:00|      7.39|    9|United States|
|2018-01-04 00:00:00|     30.05|    7|        China|
|2018-01-03 00:00:00|    179.31|    4|        China|
|2018-01-01 00:00:00|    127.89|    9|        China|
|2018-01-04 00:00:00|     43.24|    8|        China|
|2018-01-04 00:00:00|     25.76|    8|        China|
|2018-01-01 00:00:00|     166.0|    1|        China|
|2018-01-04 00:00:00|     67.51|    6|        

We can also view the schema for the table to check the data types that Spark has inferred for each column, using the `printSchema()` method:

In [4]:
sales_data.printSchema()

root
 |-- date: timestamp (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- items: integer (nullable = true)
 |-- location: string (nullable = true)



Let's run through some common transformations on the sales data.



### Filtering

Filtering can be used to quickly return a subset of the data set. Here I am using the `filter()` method to retrieve all of the orders in the United States:

In [5]:
sales_data.filter(sales_data['location'] == 'United States').show()

+-------------------+----------+-----+-------------+
|               date|total_cost|items|     location|
+-------------------+----------+-----+-------------+
|2018-01-01 00:00:00|     15.71|    2|United States|
|2018-01-06 00:00:00|    148.13|    5|United States|
|2018-01-03 00:00:00|     92.39|    2|United States|
|2018-01-05 00:00:00|      7.39|    9|United States|
|2018-01-06 00:00:00|     89.58|    7|United States|
|2018-01-02 00:00:00|      5.77|   11|United States|
|2018-01-05 00:00:00|     21.07|   12|United States|
|2018-01-04 00:00:00|     133.0|   11|United States|
|2018-01-05 00:00:00|    146.06|    4|United States|
|2018-01-02 00:00:00|    148.67|    8|United States|
|2018-01-02 00:00:00|     77.03|    3|United States|
|2018-01-03 00:00:00|     91.92|    6|United States|
|2018-01-02 00:00:00|     24.94|   11|United States|
|2018-01-01 00:00:00|     27.97|    9|United States|
|2018-01-01 00:00:00|     81.52|   12|United States|
|2018-01-05 00:00:00|    157.59|    8|United S

Spark uses a special syntax in this case to define which column I want to use when filtering `sales_data['location']`. It is similar to accessing a value of a dictionary. Unlike a dictionary which returns a single value, it returns a column that Spark will use when checking which rows match the filter condition. In this case the condition checks the location is equal to `'United States'`.

Each transformation method of a `Dataframe` returns a new `Dataframe`. I can store the result of a transformation in a variable if I want to reuse the result. Here I want to work out the number of large orders (more than 10 items):

In [6]:
large_orders = sales_data.filter(sales_data['items'] > 10)

Since I stored the result of the filter transformation in a variable I can reuse it. First let's see the number of orders:

In [7]:
large_orders.count()

181

Now let's look at first item in the data:

In [8]:
first_order = large_orders.first()
first_order

Row(date=datetime.datetime(2018, 1, 2, 0, 0), total_cost=118.46, items=12, location='China')

This returns a `Row` object, which behaves like a Python object and a Python dictionary:

In [9]:
first_order['items']

12

In [10]:
first_order.items

12

### Aggregations and Calculations

Quite often when processing data you will want to calculate a value from existing columns. You probably also want to aggregate several rows together into a single value, like a sum or an average.

Spark provides these featues in variety of different ways. 

First let's look at using the `select()` method. In this case I want to see which countries customers have placed orders from:

In [11]:
sales_data.select('location') \
    .distinct() \
    .show()

+--------------+
|      location|
+--------------+
| United States|
|         China|
|United Kingdom|
+--------------+



To start I tell Spark which columns I want to include using the `select()` method. Then I use the `distinct()` method to remove duplicate values from the result. 

Next I want to see the total sales for the data:

In [12]:
from pyspark.sql.functions import sum

sales_data.select(sum('total_cost')) \
    .show()

+-----------------+
|  sum(total_cost)|
+-----------------+
|99741.57000000005|
+-----------------+



I give the column name that I want to total 0f to the `sum()` method. This is called from inside of the `select()` method to return only the calculated value.

I can also see the average sales for the data using the `avg()` method:

In [13]:
from pyspark.sql.functions import avg

sales_data.select(avg('total_cost')) \
    .show()

+-----------------+
|  avg(total_cost)|
+-----------------+
|99.74157000000005|
+-----------------+



Sometimes I will want to calculate new columns from existing columns. For example I might want to calculate the average iten cost by dividing total order cost by the number of items:

In [14]:
sales_data \
    .withColumn('average item cost', sales_data['total_cost'] / sales_data['items']) \
    .select('date', 'location', 'Average item cost') \
    .show()

+-------------------+-------------+------------------+
|               date|     location| Average item cost|
+-------------------+-------------+------------------+
|2018-01-01 00:00:00|United States|             7.855|
|2018-01-06 00:00:00|United States|29.625999999999998|
|2018-01-06 00:00:00|        China|            10.856|
|2018-01-03 00:00:00|United States|            46.195|
|2018-01-01 00:00:00|        China|2.7883333333333336|
|2018-01-02 00:00:00|        China|            172.91|
|2018-01-05 00:00:00|        China|23.180000000000003|
|2018-01-02 00:00:00|        China|3.6166666666666667|
|2018-01-05 00:00:00|United States|0.8211111111111111|
|2018-01-04 00:00:00|        China| 4.292857142857143|
|2018-01-03 00:00:00|        China|           44.8275|
|2018-01-01 00:00:00|        China|             14.21|
|2018-01-04 00:00:00|        China|             5.405|
|2018-01-04 00:00:00|        China|              3.22|
|2018-01-01 00:00:00|        China|             166.0|
|2018-01-0

To do these kinds of calculations I need to use the dictionary-like `DataFrame` syntax. In this case I've divided the value of the `total_cost` column by the value of the `items` column. Using the `withColumn()` method I store the result of this calculation in a new column called `average item cost`.

### Grouping

Finally, let's look at grouping data. Using the `groupBy()` method I can consolidate data into related groups. After I have grouped the data I can perform aggregations and other transormations on the data.

In this example I want to see the total sales. I use the `groupBy()` method to group the data into countries:

In [15]:
from pyspark.sql.functions import format_number

country_sales = sales_data.groupBy(sales_data['location'])

country_sales.sum() \
    .select('location', format_number('sum(total_cost)', 2), 'sum(items)') \
    .withColumnRenamed('format_number(sum(total_cost), 2)', 'total sales') \
    .show()

+--------------+-----------+----------+
|      location|total sales|sum(items)|
+--------------+-----------+----------+
| United States|  10,667.63|       715|
|         China|  88,249.00|      5788|
|United Kingdom|     824.94|        30|
+--------------+-----------+----------+



After I have grouped the data by country, I calculate the sum of all the columns using the `sum()` method. I then select only the columns I want to display. The `format_number()` method is used to make sure the total sales are only displayed to two decimal places.

Applying transformations to columns gives them long and cumbersome names. I use the `withColumnRenamed()` method to make rename the calculated total cost column to `total sales`. If I didn't do this it would be called `format_number(sum(total_cost), 2)` in the output.

In this final example I want to group sales by the day of the week:

In [16]:
from pyspark.sql.functions import date_format

date_sales = sales_data.groupBy(sales_data['date'])

date_sales.sum() \
    .select(date_format('date', 'EEEE'), format_number('sum(total_cost)', 2), 'sum(items)') \
    .withColumnRenamed('date_format(date, EEEE)', 'day') \
    .withColumnRenamed('format_number(sum(total_cost), 2)', 'total sales') \
    .show()

+---------+-----------+----------+
|      day|total sales|sum(items)|
+---------+-----------+----------+
|  Tuesday|  14,403.24|       993|
|   Monday|  14,373.44|       986|
|   Friday|  17,466.03|      1035|
|Wednesday|  17,747.28|      1162|
| Saturday|  18,653.82|      1190|
| Thursday|  17,097.76|      1167|
+---------+-----------+----------+



The only new thing in this example is the use of the `date_format` function. I use it to transform the date into a human readable day of the week.

# Summary
